### Load packages

In [7]:
import pandas as pd
import numpy as np
import os
# from datetime import datetime

# import matplotlib
# import matplotlib.pyplot as plt
import networkx as nx
# from matplotlib.colors import to_rgba
#from matplotlib.patches import Arc

import plotly.graph_objects as go
# from graphviz import Digraph
# from pyvis.network import Network
# import webbrowser 

### Define parameters and file info

In [72]:
pd.set_option('display.max_colwidth', None)

In [8]:
data_directory = r"..\data"

# -----------------------------------------------------------------------
# flow data
# -----------------------------------------------------------------------
flow_csv_file = r"MCN_nonprofit_economy_revenue_2023.csv"
flow_path = os.path.join(data_directory, flow_csv_file)
print("Nonprofit economy data CSV file:", flow_path)
if os.path.exists(flow_path):
    print("EXISTS")

# -----------------------------------------------------------------------
# Output files
# -----------------------------------------------------------------------

output_directory = r"..\output"

Nonprofit economy data CSV file: ..\data\MCN_nonprofit_economy_revenue_2023.csv
EXISTS


### Load data

In [9]:
# Load nonprofit economy revenue data

flow_raw_df = pd.read_csv(flow_path)

# print(flow_raw_df.info())
flow_raw_df.head()

,Order,Source,Source Level,Recipient,Recipient Level,Amount
0,1,Donor-advised fund sponsors (national and comm...,2,"Arts, culture, humanities",3,3.42
1,2,Donor-advised fund sponsors (national and comm...,2,Education (minus colleges and universities),3,6.99
2,3,Donor-advised fund sponsors (national and comm...,2,Colleges and universities,3,6.82
3,4,Donor-advised fund sponsors (national and comm...,2,Environment and animals,3,3.41
4,5,Donor-advised fund sponsors (national and comm...,2,Health (minus hospitals and nursing homes),3,3.49


In [10]:
# Clean & format columns in revenue data

flow_clean_df = flow_raw_df.copy()

flow_clean_df["Amount"] = pd.to_numeric(
    flow_clean_df["Amount"].replace("-", 0),
    errors="coerce"
)

flow_clean_df.head(13)

,Order,Source,Source Level,Recipient,Recipient Level,Amount
0,1,Donor-advised fund sponsors (national and comm...,2,"Arts, culture, humanities",3,3.42
1,2,Donor-advised fund sponsors (national and comm...,2,Education (minus colleges and universities),3,6.99
2,3,Donor-advised fund sponsors (national and comm...,2,Colleges and universities,3,6.82
3,4,Donor-advised fund sponsors (national and comm...,2,Environment and animals,3,3.41
4,5,Donor-advised fund sponsors (national and comm...,2,Health (minus hospitals and nursing homes),3,3.49
5,6,Donor-advised fund sponsors (national and comm...,2,Hospitals and nursing homes,3,2.15
6,7,Donor-advised fund sponsors (national and comm...,2,Human Services,3,9.38
7,8,Donor-advised fund sponsors (national and comm...,2,International/foreign affairs,3,3.14
8,9,Donor-advised fund sponsors (national and comm...,2,Public/societal benefit (minus national DAF sp...,3,4.98
9,10,Donor-advised fund sponsors (national and comm...,2,Foundations (minus community foundation DAF sp...,2,0.00


### Visualize with Plotly Sankey (static charts)

Using Plotly Sankey because it includes the best compromises for:
- weighted edges
- self-loops (still poor)
- curved edges
- label control (still poor)
- heirarchy levels
- interactivity
- quality
- merging flows

In [78]:
# Separate data into self-loop and no-self-loop data sets
self_loop_df = flow_clean_df[flow_clean_df["Source"] == flow_clean_df["Recipient"]].copy()
no_self_loop_df = flow_clean_df[flow_clean_df["Source"] != flow_clean_df["Recipient"]].copy()

print("All:", len(flow_clean_df))
print("Self loops:", len(self_loop_df))
print("No self loops:", len(no_self_loop_df))

All: 130
Self loops: 2
No self loops: 128


In [80]:
# Safety copy of dataframe
plot_df = no_self_loop_df.copy()

# Build full node list
all_nodes = pd.unique(
    pd.concat([plot_df["Source"], plot_df["Recipient"]])
)

# Map node to index
node_map = {node: i for i, node in enumerate(all_nodes)}

# Convert edges to indices
sources = plot_df["Source"].map(node_map)
targets = plot_df["Recipient"].map(node_map)
values = plot_df["Amount"]

In [82]:
# Plot the Sankey diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=18,
        line=dict(color="black", width=0.5),
        label=list(all_nodes)
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values
    )
)])

# Format
fig.update_layout(
    title_text="Nonprofit Economy Flows (2023)",
    font_size=10,
    width=800,
    height=1100,
    margin=dict(l=20, r=20, t=40, b=20)
)

fig.show()

In [92]:
# Plot the Sankey diagram with self-loops in ghost mode

# Safety copy of self-loop dataframe
loop_df = self_loop_df.copy()

# Create synthetic intermediate node for each self-loop edge
loop_df["LoopNode"] = loop_df["Source"] + " (loop)"

# Expand edges into two edges per loop (through intermediate synthetic node)
loop_part1 = pd.DataFrame({
    "Source": loop_df["Source"],
    "Recipient": loop_df["LoopNode"],
    "Amount": loop_df["Amount"]
})

loop_part2 = pd.DataFrame({
    "Source": loop_df["LoopNode"],
    "Recipient": loop_df["Recipient"],
    "Amount": loop_df["Amount"]
})

# Concatenate the dataframes together
sankey_df = pd.concat([
    no_self_loop_df,
    loop_part1,
    loop_part2
], ignore_index=True)

# Create nodes
all_nodes = pd.unique(
    pd.concat([sankey_df["Source"], sankey_df["Recipient"]])
)

# Map node to index
node_map = {node: i for i, node in enumerate(all_nodes)}

# Convert edges to indices
sources = sankey_df["Source"].map(node_map)
targets = sankey_df["Recipient"].map(node_map)
values = sankey_df["Amount"]

# Plot the diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=18,
        line=dict(color="black", width=0.5),
        label=list(all_nodes)
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values
    )
)])

# Format
fig.update_layout(
    title_text="Nonprofit Economy Flows (2023)",
    font_size=10,
    width=800,
    height=1100,
    margin=dict(l=20, r=20, t=40, b=20)
)

fig.show()

### Loops, but formatted

In [ ]:
# Plot the Sankey diagram with self-loops in ghost mode

# Safety copy of self-loop dataframe
loop_df = self_loop_df.copy()

# Create synthetic intermediate node for each self-loop edge
loop_df["LoopNode"] = loop_df["Source"] + " (loop)"

# Expand edges into two edges per loop (through intermediate synthetic node)
loop_part1 = pd.DataFrame({
    "Source": loop_df["Source"],
    "Recipient": loop_df["LoopNode"],
    "Amount": loop_df["Amount"]
})

loop_part2 = pd.DataFrame({
    "Source": loop_df["LoopNode"],
    "Recipient": loop_df["Recipient"],
    "Amount": loop_df["Amount"]
})

# Concatenate the dataframes together
sankey_df = pd.concat([
    no_self_loop_df,
    loop_part1,
    loop_part2
], ignore_index=True)

# Create nodes
all_nodes = pd.unique(
    pd.concat([sankey_df["Source"], sankey_df["Recipient"]])
)

# Map node to index
node_map = {node: i for i, node in enumerate(all_nodes)}

# Convert edges to indices
sources = sankey_df["Source"].map(node_map)
targets = sankey_df["Recipient"].map(node_map)
values = sankey_df["Amount"]

# Plot the diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=18,
        line=dict(color="black", width=0.5),
        label=list(all_nodes)
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values
    )
)])

# Format
fig.update_layout(
    title_text="Nonprofit Economy Flows (2023)",
    font_size=10,
    width=800,
    height=1100,
    margin=dict(l=20, r=20, t=40, b=20)
)

fig.show()